In [0]:
%run ./00_Setup

In [0]:
from pyspark.sql.functions import current_timestamp

# Mapeamento dos caminhos de origem dos arquivos CSV na Landing Zone
data_sources = {
    "tb_base_atendentes": f"{landing_path}/base_atendentes.csv",
    "tb_base_motivos": f"{landing_path}/base_motivos.csv",
    "tb_canais": f"{landing_path}/canais.csv",
    "tb_chamados": f"{landing_path}/chamados.csv",
    "tb_chamados_hora": f"{landing_path}/Chamados_Hora.CSV",
    "tb_clientes": f"{landing_path}/clientes.csv",
    "tb_custos": f"{landing_path}/custos.csv",
    "tb_pesquisa_satisfacao": f"{landing_path}/pesquisa_satisfacao.csv"
}

print("Mapeamento de arquivos da camada bronze concluído.")

In [0]:
# Ingestão idempotente dos dados brutos com adição de metadados de auditoria
for table_name, file_path in data_sources.items():
    df_raw = spark.read.csv(file_path, header=True, inferSchema=True)
    
    df_raw \
        .withColumn("ingestion_timestamp", current_timestamp()) \
        .write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{bronze_schema}.{table_name}")

print("[SUCESSO] Ingestão da camada bronze finalizada de forma idempotente.")